# Actividad: VLSM

**Institución:** Tecnológico de Monterrey  
**Materia:** Implementación de redes de área amplia y servicios distribuidos (TC3003B)  
**Profesor:** Juan Manuel Campos Sandoval  
**Alumno:** Fernando Daniel Monroy Sánchez (A01750536)  
**Fecha:** 15 de octubre de 2026  

---

En subneteo de longitud fija (FLSM), todas las subredes adoptan el tamaño del segmento mayor, lo que ocasiona desperdicio de direcciones IP. La técnica de máscara de subred de longitud variable (VLSM) optimiza el espacio asignando bloques en potencias de 2 mediante la condición $2^h - 2 \ge N_{\text{hosts}}$, donde $h$ representa los bits de host y el prefijo CIDR resultante es $32 - h$.

## Requerimientos

La red base asignada es 172.16.0.0/16.

In [1]:
from pathlib import Path
import re
import sys

# Agregar ruta al módulo fuente
sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))

from IPython.display import display
from great_tables import GT

# Ajustar representación LaTeX de Great Tables
def _gt_to_latex(self):
    latex_str = self.as_latex()
    latex_str = re.sub(
        r"\\begin\{table\}(\[[^\]]*\])?",
        r"\\begin{table}[H]\n\\centering",
        latex_str
    )
    cols = len(self._tbl_data.columns) if hasattr(self, "_tbl_data") else 5
    if cols <= 5:
        latex_str = re.sub(
            r"\\fontsize\{[^}]+\}\{[^}]+\}\\selectfont",
            r"\\fontsize{8pt}{9.2pt}\\selectfont",
            latex_str
        )
    else:
        latex_str = re.sub(
            r"\\fontsize\{[^}]+\}\{[^}]+\}\\selectfont",
            r"\\fontsize{7.5pt}{8.8pt}\\selectfont",
            latex_str
        )
    return latex_str

GT._repr_latex_ = _gt_to_latex

from act_vlsm.network_calc import (
    get_tec_requirements,
    build_scheme_a,
    validate_no_overlap,
)
from act_vlsm.table_renderer import (
    create_requirements_df,
    create_subnets_summary_df,
    create_wan_summary_df,
)

reqs = get_tec_requirements()
df_reqs = create_requirements_df(reqs)
gt_reqs = GT(df_reqs)
display(gt_reqs)

Área Académica,Hosts Req.,Bits h,Hosts Útiles,Máscara / Prefijo
Prepa Tec,"8,000",13,"8,190",255.255.224.0 (/19)
Ingeniería Tec,"4,000",12,"4,094",255.255.240.0 (/20)
Negocios Tec,"4,000",12,"4,094",255.255.240.0 (/20)
Estudios Creativos,"2,000",11,"2,046",255.255.248.0 (/21)
Ciencias Médicas,"1,000",10,"1,022",255.255.252.0 (/22)
Humanidades,500,9,510,255.255.254.0 (/23)
Enlaces WAN (6),2 c/u (6 enlaces),2,2 c/u,255.255.255.252 (/30)


## Subredes

Asignación modular de subredes para las 6 áreas académicas dentro de los bloques iniciales /19:

In [2]:
lan_subnets, wan_subnets = build_scheme_a()
validate_no_overlap(lan_subnets + wan_subnets)

df_lan = create_subnets_summary_df(lan_subnets)
gt_lan = GT(df_lan)
display(gt_lan)

Área Académica,IP de red,Rango Útil,Broadcast,Máscara,Req.,Útiles,Libres
Prepa Tec,172.16.0.0,172.16.0.1 a 172.16.31.254,172.16.31.255,255.255.224.0 (/19),"8,000","8,190",190
Ingeniería Tec,172.16.32.0,172.16.32.1 a 172.16.47.254,172.16.47.255,255.255.240.0 (/20),"4,000","4,094",94
Negocios Tec,172.16.64.0,172.16.64.1 a 172.16.79.254,172.16.79.255,255.255.240.0 (/20),"4,000","4,094",94
Estudios Creativos,172.16.96.0,172.16.96.1 a 172.16.103.254,172.16.103.255,255.255.248.0 (/21),"2,000","2,046",46
Ciencias Médicas,172.16.128.0,172.16.128.1 a 172.16.131.254,172.16.131.255,255.255.252.0 (/22),"1,000","1,022",22
Humanidades,172.16.160.0,172.16.160.1 a 172.16.161.254,172.16.161.255,255.255.254.0 (/23),500,510,10


## Enlaces

Asignación de 6 enlaces punto a punto en el extremo superior del bloque 172.16.224.0/19:

In [3]:
df_wan = create_wan_summary_df(wan_subnets)
gt_wan = GT(df_wan)
display(gt_wan)

Enlace WAN,IP de red,Rango Útil,Broadcast,Máscara,Req.,Útiles,Libres
WAN 1 (Prepa),172.16.255.232,172.16.255.233 a 172.16.255.234,172.16.255.235,255.255.255.252 (/30),2,2,0
WAN 2 (Ingeniería),172.16.255.236,172.16.255.237 a 172.16.255.238,172.16.255.239,255.255.255.252 (/30),2,2,0
WAN 3 (Negocios),172.16.255.240,172.16.255.241 a 172.16.255.242,172.16.255.243,255.255.255.252 (/30),2,2,0
WAN 4 (Creativos),172.16.255.244,172.16.255.245 a 172.16.255.246,172.16.255.247,255.255.255.252 (/30),2,2,0
WAN 5 (Médicas),172.16.255.248,172.16.255.249 a 172.16.255.250,172.16.255.251,255.255.255.252 (/30),2,2,0
WAN 6 (Humanidades),172.16.255.252,172.16.255.253 a 172.16.255.254,172.16.255.255,255.255.255.252 (/30),2,2,0
